# NLP Full Course Notebook

This Colab notebook is an original companion **Natural Language Processing with Python** lecture.

## What you will cover
- Setup and library installation
- Text preprocessing
- Tokenization, stopwords, stemming, lemmatization
- Bag of Words and TF-IDF
- Basic text classification with scikit-learn
- Named Entity Recognition with spaCy
- Transformer pipelines with Hugging Face

> If you open this in **Google Colab**, run the setup cells first.


In [1]:
# Install core NLP packages in Colab
# - nltk: Natural Language Toolkit for text processing.
# - spacy: Industrial-strength Natural Language Processing.
# - scikit-learn: Machine learning library for Python.
# - textblob: Simplified text processing library.
# - transformers: Hugging Face library for state-of-the-art NLP models.
# - datasets: Hugging Face library for easy access to datasets.
# - sentencepiece: SentencePiece for subword tokenization.
#!pip -q install nltk spacy scikit-learn textblob transformers datasets sentencepiece

# Download the small English language model for spaCy
# This model provides capabilities like tokenization, POS tagging, NER, etc.
#!python -m spacy download en_core_web_sm -q

In [2]:
import re # Regular expression operations
import numpy as np # Numerical computing library
import pandas as pd # Data manipulation and analysis library
import nltk # Natural Language Toolkit
import spacy # Industrial-strength Natural Language Processing

from pprint import pprint # For pretty-printing data structures
from nltk.corpus import stopwords # To get a list of common stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer # For stemming and lemmatization
from nltk.tokenize import word_tokenize, sent_tokenize # For tokenization
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer # For text vectorization
from sklearn.model_selection import train_test_split # For splitting data into training and testing sets
from sklearn.pipeline import Pipeline # To create a pipeline of transformers and an estimator
from sklearn.linear_model import LogisticRegression # Classification model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # For evaluating model performance
from textblob import TextBlob # For simplified text processing and sentiment analysis

# Download necessary NLTK data
# - 'punkt': Pre-trained tokenizer for English.
# - 'punkt_tab': Tokenizer for tabular data (less common).
# - 'stopwords': List of common stop words.
# - 'wordnet': Lexical database for English (used by WordNetLemmatizer).
# - 'omw-1.4': Open Multilingual Wordnet (dependency for wordnet).
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load the English language model for spaCy
# This model provides linguistic annotations like parts-of-speech, named entities, etc.
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 1. What is NLP?

Natural Language Processing helps computers work with human language in text or speech.

Common tasks:
- text classification
- sentiment analysis
- named entity recognition
- machine translation
- summarization
- question answering


In [3]:
# Define a sample text for NLP demonstrations
sample_text = '''
NLP is one of the most exciting areas of AI.
It helps machines understand language, extract meaning, and generate responses.
Popular tools include NLTK, spaCy, scikit-learn, and Hugging Face Transformers.
'''
# Print the sample text
print(sample_text)


NLP is one of the most exciting areas of AI.
It helps machines understand language, extract meaning, and generate responses.
Popular tools include NLTK, spaCy, scikit-learn, and Hugging Face Transformers.



## 2. Sentence and word tokenization

In [4]:
# Perform sentence tokenization: splitting the text into individual sentences
sentences = sent_tokenize(sample_text)
# Perform word tokenization: splitting the text into individual words
words = word_tokenize(sample_text)

# Print the tokenized sentences
print("Sentences:")
pprint(sentences)
# Print the first 25 tokenized words
print("\nWords:")
pprint(words[:25])

Sentences:
['\nNLP is one of the most exciting areas of AI.',
 'It helps machines understand language, extract meaning, and generate '
 'responses.',
 'Popular tools include NLTK, spaCy, scikit-learn, and Hugging Face '
 'Transformers.']

Words:
['NLP',
 'is',
 'one',
 'of',
 'the',
 'most',
 'exciting',
 'areas',
 'of',
 'AI',
 '.',
 'It',
 'helps',
 'machines',
 'understand',
 'language',
 ',',
 'extract',
 'meaning',
 ',',
 'and',
 'generate',
 'responses',
 '.',
 'Popular']


## 3. Cleaning text

In [5]:
# Define a function to clean text
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove non-alphabetic characters (keep only letters and spaces)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    # Replace multiple spaces with a single space and strip leading/trailing spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Clean the sample text
cleaned = clean_text(sample_text)
# Print the cleaned text
print(cleaned)

nlp is one of the most exciting areas of ai it helps machines understand language extract meaning and generate responses popular tools include nltk spacy scikitlearn and hugging face transformers


## 4. Stopword removal, stemming, lemmatization

In [6]:
# Get a set of English stop words (common words like 'the', 'is', 'a')
stop_words = set(stopwords.words("english"))
# Initialize the Porter Stemmer for stemming words
stemmer = PorterStemmer()
# Initialize the WordNet Lemmatizer for lemmatizing words
lemmatizer = WordNetLemmatizer()

# Tokenize the cleaned text into individual words
tokens = word_tokenize(cleaned)
# Remove stop words from the tokens
tokens_wo_stop = [t for t in tokens if t not in stop_words]
# Apply stemming to the tokens without stop words
stemmed = [stemmer.stem(t) for t in tokens_wo_stop]
# Apply lemmatization to the tokens without stop words
lemmatized = [lemmatizer.lemmatize(t) for t in tokens_wo_stop]

# Print the different stages of text processing
print("Original tokens:", tokens)
print("\nWithout stopwords:", tokens_wo_stop)
print("\nStemmed:", stemmed)
print("\nLemmatized:", lemmatized)

Original tokens: ['nlp', 'is', 'one', 'of', 'the', 'most', 'exciting', 'areas', 'of', 'ai', 'it', 'helps', 'machines', 'understand', 'language', 'extract', 'meaning', 'and', 'generate', 'responses', 'popular', 'tools', 'include', 'nltk', 'spacy', 'scikitlearn', 'and', 'hugging', 'face', 'transformers']

Without stopwords: ['nlp', 'one', 'exciting', 'areas', 'ai', 'helps', 'machines', 'understand', 'language', 'extract', 'meaning', 'generate', 'responses', 'popular', 'tools', 'include', 'nltk', 'spacy', 'scikitlearn', 'hugging', 'face', 'transformers']

Stemmed: ['nlp', 'one', 'excit', 'area', 'ai', 'help', 'machin', 'understand', 'languag', 'extract', 'mean', 'gener', 'respons', 'popular', 'tool', 'includ', 'nltk', 'spaci', 'scikitlearn', 'hug', 'face', 'transform']

Lemmatized: ['nlp', 'one', 'exciting', 'area', 'ai', 'help', 'machine', 'understand', 'language', 'extract', 'meaning', 'generate', 'response', 'popular', 'tool', 'include', 'nltk', 'spacy', 'scikitlearn', 'hugging', 'face

## 5. Bag of Words and TF-IDF

In [7]:
# Define a list of sample documents
docs = [
    "I love this course on natural language processing",
    "This NLP course is practical and useful",
    "I dislike boring lectures",
    "The class project uses Python and transformers"
]

# Initialize CountVectorizer to convert a collection of text documents to a matrix of token counts.
# ngram_range=(1, 2) means it will consider unigrams (single words) and bigrams (two-word phrases).
count_vec = CountVectorizer(ngram_range=(1, 2))
# Fit the vectorizer to the documents and transform them into a Bag of Words matrix
X_count = count_vec.fit_transform(docs)

# Initialize TfidfVectorizer to convert a collection of raw documents to a matrix of TF-IDF features.
tfidf_vec = TfidfVectorizer()
# Fit the vectorizer to the documents and transform them into a TF-IDF matrix
X_tfidf = tfidf_vec.fit_transform(docs)

# Print the shape of the Bag of Words matrix (number of documents, number of features)
print("CountVectorizer shape:", X_count.shape)
# Print a sample of the feature names generated by CountVectorizer
print("Sample BoW features:", count_vec.get_feature_names_out()[:20])

# Print the shape of the TF-IDF matrix
print("\nTF-IDF shape:", X_tfidf.shape)
# Print a sample of the feature names generated by TfidfVectorizer
print("Sample TF-IDF features:", tfidf_vec.get_feature_names_out()[:20])

CountVectorizer shape: (4, 41)
Sample BoW features: ['and' 'and transformers' 'and useful' 'boring' 'boring lectures' 'class'
 'class project' 'course' 'course is' 'course on' 'dislike'
 'dislike boring' 'is' 'is practical' 'language' 'language processing'
 'lectures' 'love' 'love this' 'natural']

TF-IDF shape: (4, 21)
Sample TF-IDF features: ['and' 'boring' 'class' 'course' 'dislike' 'is' 'language' 'lectures'
 'love' 'natural' 'nlp' 'on' 'practical' 'processing' 'project' 'python'
 'the' 'this' 'transformers' 'useful']


In [8]:
# Create a Pandas DataFrame from the Bag of Words matrix
# X_count.toarray() converts the sparse matrix to a dense NumPy array.
# columns are set to the feature names obtained from the CountVectorizer.
bow_df = pd.DataFrame(
    X_count.toarray(),
    columns=count_vec.get_feature_names_out()
)
# Display the first 10 rows of the DataFrame
bow_df.head(10)

,and,and transformers,and useful,boring,boring lectures,class,class project,course,course is,course on,...,python and,the,the class,this,this course,this nlp,transformers,useful,uses,uses python
0,0,0,0,0,0,0,0,1,0,1,...,0,0,0,1,1,0,0,0,0,0
1,1,0,1,0,0,0,0,1,1,0,...,0,0,0,1,0,1,0,1,0,0
2,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1,0,0,0,1,1,0,0,0,...,1,1,1,0,0,0,1,0,1,1


## 6. Mini sentiment analysis demo

In [9]:
# Define a list of reviews for sentiment analysis
reviews = [
    "This lecture is amazing and very clear.",
    "I really enjoyed the examples and the teaching style.",
    "The session was too long and confusing.",
    "The notebook is practical and easy to follow.",
    "I do not like the pacing of this class."
]

# Iterate through each review
for review in reviews:
    # Create a TextBlob object for the current review
    blob = TextBlob(review)
    # Print the review and its sentiment polarity and subjectivity
    # Polarity ranges from -1 (negative) to 1 (positive).
    # Subjectivity ranges from 0 (objective) to 1 (subjective).
    print(f"{review}\nPolarity: {blob.sentiment.polarity:.3f}, Subjectivity: {blob.sentiment.subjectivity:.3f}\n")

This lecture is amazing and very clear.
Polarity: 0.365, Subjectivity: 0.699

I really enjoyed the examples and the teaching style.
Polarity: 0.500, Subjectivity: 0.700

The session was too long and confusing.
Polarity: -0.175, Subjectivity: 0.400

The notebook is practical and easy to follow.
Polarity: 0.433, Subjectivity: 0.833

I do not like the pacing of this class.
Polarity: 0.000, Subjectivity: 0.000



## 7. Text classification with scikit-learn

Below is a tiny toy dataset just to demonstrate the workflow.


In [10]:
# Create a Pandas DataFrame with sample text data and labels
# 'text' column contains the messages.
# 'label' column indicates whether the message is spam/promo (1) or normal (0).
data = pd.DataFrame({
    "text": [
        "Win money now claim your free prize",
        "Lowest price available buy today",
        "Are we meeting tomorrow morning",
        "Please review the attached project report",
        "Exclusive offer limited time discount",
        "Let's schedule the client call for Friday",
        "Claim your voucher and get rewards",
        "Can you send the meeting notes"
    ],
    "label": [1, 1, 0, 0, 1, 0, 1, 0]  # 1 = spam/promo, 0 = normal
})

# Split the data into training and testing sets
# X_train, X_test: features for training and testing.
# y_train, y_test: labels for training and testing.
# test_size=0.25: 25% of the data will be used for testing.
# random_state=42: for reproducibility of the split.
# stratify=data["label"]: ensures that the proportion of labels is the same in both train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    data["text"], data["label"], test_size=0.25, random_state=42, stratify=data["label"]
)

# Create a machine learning pipeline
# The pipeline first applies TF-IDF vectorization and then trains a Logistic Regression model.
clf = Pipeline([
    ("tfidf", TfidfVectorizer()), # Step 1: Convert text to TF-IDF features
    ("model", LogisticRegression(max_iter=500)) # Step 2: Apply Logistic Regression model
])

# Train the classifier using the training data
clf.fit(X_train, y_train)
# Make predictions on the test data
preds = clf.predict(X_test)

# Evaluate the model performance
# Calculate and print the accuracy score
print("Accuracy:", accuracy_score(y_test, preds))
# Print a detailed classification report (precision, recall, f1-score, support)
# zero_division=0: handles cases where precision/recall is 0 to avoid warnings.
print("\nClassification report:")
print(classification_report(y_test, preds, zero_division=0))

Accuracy: 1.0

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



## 8. Named Entity Recognition with spaCy

In [11]:
# Process a sample sentence using the loaded spaCy NLP model
doc = nlp("OpenAI released GPT models in San Francisco and developers use Python worldwide.")
# Extract named entities from the processed document
# Each entity is represented as a tuple: (entity text, entity label)
# e.g., ('OpenAI', 'ORG') means 'OpenAI' is an Organization.
[(ent.text, ent.label_) for ent in doc.ents]

[('OpenAI', 'ORG'), ('GPT', 'ORG'), ('San Francisco', 'GPE')]

In [12]:
# Import displacy for visual display of named entities
from spacy import displacy
# Render the document with named entities highlighted in a Jupyter/Colab environment
# style="ent" specifies that entities should be highlighted.
# jupyter=True enables rendering specifically for Jupyter notebooks.
displacy.render(doc, style="ent", jupyter=True)

## 9. Transformer pipelines with Hugging Face

In [13]:
# Import the pipeline function from the transformers library
from transformers import pipeline

# Create a sentiment analysis pipeline
# If no model is specified, it defaults to a pre-trained model like 'distilbert-base-uncased-finetuned-sst-2-english'.
sentiment_pipe = pipeline("sentiment-analysis")
# Use the pipeline to analyze the sentiment of two example sentences
sentiment_pipe([
    "This NLP workshop was excellent.",
    "The examples were hard to follow."
])

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\MiniForge\envs\kpitb_env\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Saif Ullah\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9997890591621399},
 {'label': 'NEGATIVE', 'score': 0.999468982219696}]

In [15]:
# Optional: summarization may download a larger model
# This code block demonstrates how to use a summarization pipeline from Hugging Face Transformers.
# summary_pipe = pipeline("summarization") # Initialize the summarization pipeline.
# summary_pipe("""Natural Language Processing is a field of AI focused on helping computers
# understand, analyze, and generate human language. It includes preprocessing, vectorization,
# modeling, evaluation, and deployment.""", max_length=40, min_length=15) # Generate a summary with specified length constraints.

## 10. Suggested exercises

1. Replace the toy dataset with your own CSV file.
2. Compare CountVectorizer vs TF-IDF.
3. Try a different classifier such as LinearSVC or Naive Bayes.
4. Add bigrams and compare accuracy.
5. Run a transformer model for zero-shot classification.
6. Build a small end-to-end sentiment classifier project.


## 11. Suggested project ideas

- movie review sentiment analysis
- email spam classifier
- support ticket intent classification
- news topic classifier
- resume keyword extractor
- chatbot FAQ retriever
